# Spatial analysis and islet inflammation

Inspect a whole pancreatic section, measure cell neighborhoods from native centroids, and compare islet pseudotime with core inflammation across donors.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/03_spatial_analysis.ipynb).

## 1. Select a donor

The native-cell graph connects cells within a section. The islet graph has a separate fit for each donor.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "pancreas_islets.h5ad").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the processed PhenoCycler data directory.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})
DONOR = "6533"                      # Any donor listed in the manifest
DISEASE = "T1D"                     # ND, AAb+, or T1D
LEVEL = "primary"                    # primary, detail, immune_class, t_subtype, macrophage_profile
MARKERS = ["GCG","INS","SST","CD99","CD3e","CD20","CD68","CD31","EpCAM"]
PAIR = ("INS","SST")
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
type_order = ["Alpha","Beta","Delta","Endocrine (ambiguous subtype)","Endocrine (unresolved subtype)",
              "Exocrine","Immune","Blood vessel","Lymphatic","Nerve","Unresolved (no lineage evidence)"]
inks = {"Alpha":"#cf6c30", "Beta":"#287cac", "Delta":"#4c8b58", "Immune":"#965084",
        "Exocrine":"#c1b7a6", "Endocrine (ambiguous subtype)":"#bda559",
        "Endocrine (unresolved subtype)":"#99853f", "Blood vessel":"#b34b50",
        "Lymphatic":"#68a69a", "Nerve":"#5b558c", "Unresolved (no lineage evidence)":"#868686"}
for label in type_order:
    inks.setdefault(str(label), "#73777d")

## 2. Spatial visualization

Cell identities and islet compartments are shown at their native centroids using a uniform display sample.

In [ ]:
# Native coordinates, µm. Draw a uniform subset of centroids to keep whole sections legible.
donor_path = data / "pancreas" / f"{DONOR}.h5ad"
donor = sc.read_h5ad(donor_path,backed="r")
assert donor.n_obs == manifest.set_index("donor").at[DONOR,"cells"]
rng = np.random.default_rng(19)
ix = np.sort(rng.choice(donor.n_obs,min(donor.n_obs,60000),replace=False))
obs = donor.obs.iloc[ix][["celltype","primary","islet_region"]].copy()
xy = np.asarray(donor.obsm["spatial"])[ix]
section_sample = ad.AnnData(obs=obs, obsm={"spatial":xy})
fig, axes = plt.subplots(1,2,figsize=(12,5.8))
zones = {"outside":"#c8c6c1","core":"#126891","peri":"#d78939","multiple peri-islet bands":"#6e4c88"}
for ax,key,palette,title in [(axes[0],"celltype",inks,"Cell identity"),(axes[1],"islet_region",zones,"Native islet compartments")]:
    sc.pl.embedding(section_sample,basis="spatial",color=key,ax=ax,show=False,
                    size=3,palette=palette,legend_loc="none",title=title)
    ax.set_aspect("equal"); ax.invert_yaxis()
    labels=section_sample.obs[key].cat.categories
    handles=[Line2D([],[],marker="o",ms=4,ls="",color=palette[t],label=t) for t in labels]
    ax.legend(handles=handles,loc="upper center",bbox_to_anchor=(.5,-.02),ncol=2,frameon=False,fontsize=7)
fig.suptitle(f"Donor {DONOR} · {donor.n_obs:,} admitted cells · {len(ix):,} uniformly sampled centroids",y=.99)
fig.subplots_adjust(bottom=.26,top=.88,wspace=.17)
plt.show()

## 3. Spatial neighborhood enrichment

Each row is a focal phenotype and each column is a neighbor phenotype. The graph uses every admitted centroid in the selected section. The heatmap shows descriptive enrichment relative to the section abundance. Gray cells have no observed edges or an undefined expectation.

In [ ]:
from scipy.spatial import cKDTree

# Ten nearest other cells, computed within this section from all admitted native centroids.
xy = np.asarray(donor.obsm["spatial"], dtype=np.float64)
labels = donor.obs["celltype"].astype("category").cat.remove_unused_categories()
groups = labels.cat.categories
codes = labels.cat.codes.to_numpy()
tree = cKDTree(xy)
k = min(10, len(xy)-1)
pairs = np.zeros((len(groups),len(groups)),dtype=np.int64)
for start in range(0,len(xy),50000):
    stop = min(start+50000,len(xy))
    _, neighbors = tree.query(xy[start:stop],k=k+1, workers=2)
    centers = np.arange(start,stop)[:,None]
    rank = np.cumsum(neighbors!=centers,axis=1)
    take = (neighbors!=centers) & (rank<=k)
    edges = codes[start:stop,None]*len(groups) + codes[neighbors]
    pairs += np.bincount(edges[take],minlength=len(groups)**2).reshape(pairs.shape)
assert pairs.sum() == len(xy)*k
# Enrichment relative to random mixing at the observed section-wide abundance.
# A cell cannot be its own neighbor; use the finite-population expectation.
abundance = np.bincount(codes,minlength=len(groups))
null = (abundance[None,:] - np.eye(len(groups))) / (len(xy)-1)
pobs = pairs / pairs.sum(axis=1,keepdims=True)
ratio = np.log2(np.divide(pobs,null,out=np.full_like(pobs,np.nan),where=(pobs>0)&(null>0)))
order = [groups.get_loc(t) for t in type_order if t in groups]
mat = ratio[np.ix_(order,order)]; names=groups[order].tolist()
fig, ax = plt.subplots(figsize=(7.4,5.2))
lim=max(1,float(np.nanpercentile(np.abs(mat),95)))
cmap=plt.get_cmap("RdBu_r").copy();cmap.set_bad("#dadada")
im=ax.imshow(mat,cmap=cmap,norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim),aspect="auto")
ax.set(xticks=range(len(names)),yticks=range(len(names)),xticklabels=names,yticklabels=names,
       title=f"Cell neighborhoods · donor {DONOR}",xlabel="Neighbor identity",ylabel="Focal cell identity")
plt.setp(ax.get_xticklabels(),rotation=45,ha="right")
fig.colorbar(im,ax=ax,label=r"$\log_2$(observed / expected)",shrink=.72)
fig.tight_layout();plt.show()

## 4. Per-donor pseudotime and measured islet inflammation

Each point is one native islet. The first two panels share graph coordinates. Core inflammation is measured in immune cells/mm². Lower and upper positive burden groups split at the donor’s median positive density across occupied cores.

In [ ]:
islets = sc.read_h5ad(data / "pancreas_islets.h5ad")
section = islets[islets.obs.donor.astype(str).eq(DONOR)].copy()
keep = np.isfinite(section.obs.pseudotime_rank) & np.isfinite(section.obsm["X_islet_graph"]).all(axis=1)
fit = section[keep].copy()
print(f"{DONOR} · {section.n_obs:,} native islets · {fit.n_obs:,} with pseudotime")
if fit.n_obs:
    fig, axes = plt.subplots(1,3,figsize=(13.5,3.9))
    for ax, field, title, cmap in [(axes[0],"pseudotime_rank","Within-donor pseudotime","viridis"),
                                   (axes[1],"core_immune_per_mm2","Core immune cells/mm²","magma")]:
        sc.pl.embedding(fit,basis="islet_graph",color=field,cmap=cmap,ax=ax,show=False,
                        colorbar_loc="right",size=24,title=title,frameon=False)
    sc.pl.scatter(fit,x="pseudotime_rank",y="core_immune_per_mm2",color="burden",
                  palette=[{"Zero":"#959a9b","Lower":"#43868b","Upper":"#a84238"}[c] for c in fit.obs.burden.cat.categories],
                  ax=axes[2],show=False,size=14,legend_loc="none",frameon=True)
    axes[2].set(title="Inflammation along pseudotime",xlabel="Within-donor pseudotime rank",ylabel="Core immune cells/mm²")
    axes[2].legend(handles=[Line2D([],[],marker="o",ls="",color=k,label=n) for n,k in
                         [("Zero","#959a9b"),("Lower","#43868b"),("Upper","#a84238")]],
                   frameon=False,fontsize=8)
    fig.tight_layout()
    plt.show()
display(section.obs.groupby("burden",observed=True).agg(native_occupied_islets=("islet_id","size"),
        islets_with_pseudotime=("pseudotime_rank","count"), median_core_immune_per_mm2=("core_immune_per_mm2","median"),
        median_pseudotime_rank=("pseudotime_rank","median")).round(3))

## 5. Across-donor view

Each point summarizes one donor. Spearman correlation measures the relationship between that donor’s islet pseudotime and measured core inflammation.

In [ ]:
res = []
for (group, did), frame in islets.obs.groupby(["disease","donor"],observed=True):
    z=frame.dropna(subset=["pseudotime_rank","core_immune_per_mm2"])
    rho=z.pseudotime_rank.corr(z.core_immune_per_mm2,method="spearman") if z.core_immune_per_mm2.nunique()>1 else np.nan
    res.append({"disease":group,"donor":did,"native_islets":len(frame),"fitted":len(z),"rho":rho})
overview=pd.DataFrame(res).set_index("donor")
fig,ax=plt.subplots(figsize=(7,3.8))
for i,g in enumerate(group_order):
    y=overview.loc[overview.disease.eq(g),"rho"].dropna()
    ax.scatter(i+np.linspace(-.15,.15,len(y)),y,color=group_ink[g],s=30)
    ax.hlines(y.median(),i-.23,i+.23,color="#333")
ax.axhline(0,color="#777",lw=.7);ax.set(xticks=range(3),xticklabels=group_order,ylim=(-1,1),ylabel="Within-donor Spearman ρ")
fig.tight_layout();plt.show()
display(overview)